# NB0 — Setup & the Model

Goal of this notebook: get GPT-2 small running as a `HookedTransformer` and get fluent with the
four things you'll do constantly for the rest of the course:

1. **Load** the model.
2. **Tokenize** text (and turn tokens back into readable strings).
3. **Run** a forward pass and read the **logits**.
4. **Inspect** `model.cfg` — the shapes and hyper-parameters that every later notebook depends on.

You already know attention internals, so we won't re-derive them. TransformerLens (TL) is just a
transformer implementation that exposes every internal activation through *hooks* — we start using
those in NB1. Here we just get the model breathing.

## 1. Setup and load

We disable gradients globally: this whole course is *analysis*, not training, so we never need the
autograd graph. That saves memory and makes everything faster.

In [2]:
import torch
from transformer_lens import HookedTransformer

# We are analysing, not training — turn off autograd everywhere.
torch.set_grad_enabled(False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# from_pretrained downloads (or reuses the cached) GPT-2 weights and folds them into TL's
# HookedTransformer format. The model is placed on `device` automatically.
model = HookedTransformer.from_pretrained("gpt2", device=device)
print("loaded:", model.cfg.model_name)

device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
loaded: gpt2


## 2. Tokenization

GPT-2 uses byte-pair encoding (BPE). Two helpers you'll use all the time:

- `model.to_tokens(text)` → a `[batch, position]` `LongTensor` of token **ids**. By default it
  prepends the *beginning-of-sequence* (BOS) token, which for GPT-2 is `<|endoftext|>`.
- `model.to_str_tokens(text)` → the same sequence as a **list of strings**, so you can see exactly
  how the text was split. This is invaluable when you're reasoning about which position is which.

Notice that BPE tokens often include a leading space (e.g. `' cat'`), and a single word can split
into several tokens.

In [3]:
text = "The cat sat on the mat"

tokens = model.to_tokens(text)
str_tokens = model.to_str_tokens(text)

print("token ids shape:", tokens.shape)   # [1, seq_len]  (batch of 1)
print("token ids:", tokens)
print("str tokens:", str_tokens)          # note the BOS token and the leading spaces
print("n tokens:", len(str_tokens))

token ids shape: torch.Size([1, 7])
token ids: tensor([[50256,   464,  3797,  3332,   319,   262,  2603]], device='cuda:0')
str tokens: ['<|endoftext|>', 'The', ' cat', ' sat', ' on', ' the', ' mat']
n tokens: 7


## 3. Forward pass and logits

Calling `model(tokens)` runs the forward pass and returns **logits** of shape
`[batch, position, d_vocab]`. The logits at position `i` are the model's prediction for the token at
position `i+1` — i.e. "what comes next" given everything up to and including position `i`.

So to see what the model predicts follows the *whole* prompt, we look at the logits at the **last**
position, `logits[0, -1]`, and take the arg-max.

In [4]:
logits = model(tokens)
print("logits shape:", logits.shape)   # [1, seq_len, d_vocab]; d_vocab = 50257 for GPT-2

# Prediction for the token AFTER the last input token:
last_logits = logits[0, -1]            # shape [d_vocab]
next_id = last_logits.argmax()
print("top next-token id:", next_id.item())
print("top next-token str:", repr(model.to_string(next_id)))

# Top-5 predictions, as a quick sanity check that the model is sane.
top5 = last_logits.topk(5)
for value, idx in zip(top5.values, top5.indices):
    print(f"  {repr(model.to_string(idx)):>12}   logit={value.item():.2f}")

logits shape: torch.Size([1, 7, 50257])
top next-token id: 11
top next-token str: ','
           ','   logit=15.79
        ' and'   logit=15.40
         ' in'   logit=14.90
       ' with'   logit=14.87
        ' for'   logit=14.84


## 4. Generation

`model.generate` samples autoregressively. For reproducible, deterministic output we pass
`do_sample=False` (greedy — always take the arg-max). This is a nice end-to-end sanity check that
the whole pipeline works.

In [5]:
out = model.generate(
    "The Eiffel Tower is located in the city of",
    max_new_tokens=10,
    do_sample=False,      # greedy decoding -> deterministic
    verbose=False,
)
print(out)

The Eiffel Tower is located in the city of London, and is the tallest building in the world


## 5. The model config

`model.cfg` holds every architectural number you'll refer to later. Commit the big four to memory:

- `n_layers` — number of transformer blocks (12).
- `n_heads` — attention heads per block (12).
- `d_model` — width of the residual stream (768).
- `d_head` — dimension per head (64); note `n_heads * d_head == d_model`.

`n_layers * n_heads = 144` total attention heads — that's the search space we'll hunt through for
induction heads in NB3.

In [6]:
cfg = model.cfg
print("n_layers :", cfg.n_layers)
print("n_heads  :", cfg.n_heads)
print("d_model  :", cfg.d_model)
print("d_head   :", cfg.d_head)
print("d_vocab  :", cfg.d_vocab)
print("n_ctx    :", cfg.n_ctx)
print("total attention heads:", cfg.n_layers * cfg.n_heads)

n_layers : 12
n_heads  : 12
d_model  : 768
d_head   : 64
d_vocab  : 50257
n_ctx    : 1024
total attention heads: 144


## Practice

Now your turn. Fill in the `TODO(you)` cell below. The task:

> Given the prompt `"When John and Mary went to the shops, John gave a drink to"`, compute the
> model's **most likely next token** and its **probability** (not just the logit).

Hints:
- Tokenize with `model.to_tokens`.
- Run the forward pass and take the logits at the **last** position.
- Turn logits into probabilities with `logits.softmax(dim=-1)`.
- Use `.argmax()` for the id, `model.to_string(...)` for the string, and index the prob tensor.

(This particular prompt is the classic "IOI" sentence — a well-known GPT-2 behaviour. A good model
should strongly prefer `' Mary'`.)

In [8]:
prompt = "When John and Mary went to the shops, John gave a drink to"

tokens = model.to_tokens(prompt)
logits = model(tokens)
probs = logits[0,-1].softmax(dim=-1)
top_id = probs.argmax()
print("top token:",repr(model.to_string(top_id)))
print("probrbility:",probs[top_id].item())
# TODO(you): compute the top next token and its probability.
# tokens = ...
# logits = ...
# probs  = ...
# top_id = ...
# print(top token string, top probability)


top token: ' Mary'
probrbility: 0.4947144091129303


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
tokens = model.to_tokens(prompt)
logits = model(tokens)
probs = logits[0, -1].softmax(dim=-1)
top_id = probs.argmax()
print("top token:", repr(model.to_string(top_id)))
print("probability:", probs[top_id].item())
```

</details>

---
**Done?** When your practice cell prints `' Mary'` with a high probability, ask me to review it, and
we'll move on to **NB1 — hooks & the activation cache**, where TransformerLens starts to earn its
keep.